# M9.2 · NIM Deployment (~20–25 min)

**All steps in this notebook:** pip, GPU, NGC key prompt, catalog NIM pull/run, optional custom Model-Free NIM from M7 SFT.

Production reference: `gsi-training/9.custom_model_deployment/README.md` (8× GPU, TP=2). Workshop: **1× GPU, TP=1**.


## 1. Prerequisites (pip + NGC + NIM)


In [1]:
import sys
from pathlib import Path

# Workshop shared helpers (parent folder: workshop-Materials/).
sys.path.insert(0, str(Path.cwd().parent))
from docker_storage import ensure_docker_storage
from notebook_env import bootstrap_notebook_env, ensure

ensure_docker_storage()  # docker images/layers on /data — avoids root disk full errors
bootstrap_notebook_env()

# M9 dependencies — installed inline so this notebook is fully self-contained.
for mod, pkg in [
    ("torch", "torch>=2.5.0,<2.7.0"),
    ("transformers", "transformers>=4.45.0,<4.55.0"),
    ("peft", "peft>=0.13.0,<0.16.0"),
    ("requests", "requests>=2.32.0"),
]:
    ensure(mod, [pkg], quiet=True)
print("Prerequisites ready.")


2026-06-12 07:32:04,893 INFO === ensure_docker_storage (log: /data/logs/docker_storage.log) ===
2026-06-12 07:32:04,894 INFO disk /: 17.4G used / 123.9G (14.1%)
2026-06-12 07:32:04,894 INFO disk /data: 0.0G used / 502.9G (477.3G free)
2026-06-12 07:32:04,895 INFO env TMPDIR=/data/cache/tmp
2026-06-12 07:32:04,895 INFO env DOCKER_TMPDIR=/data/cache/tmp
2026-06-12 07:32:04,896 INFO env PIP_CACHE_DIR=/data/cache/pip
2026-06-12 07:32:04,896 INFO env UV_CACHE_DIR=/data/cache/uv
2026-06-12 07:32:04,897 INFO env HF_HOME=/data/cache/hf
2026-06-12 07:32:04,897 INFO env XDG_CACHE_HOME=/data/cache/xdg
2026-06-12 07:32:04,898 INFO env LOCAL_NIM_CACHE=/data/cache/nim
2026-06-12 07:32:04,898 INFO $ docker info --format {{.DockerRootDir}}
2026-06-12 07:32:04,940 INFO docker data-root (config): None
2026-06-12 07:32:04,941 INFO docker data-root (live):   /var/lib/docker
2026-06-12 07:32:04,941 INFO migrating docker storage to /data/docker ...
2026-06-12 07:32:04,941 INFO stopping docker ...
2026-06-12

docker storage ok: /data/docker (477.3G free on volume)
creating uv venv: /home/ubuntu/workshop-materials-gsi/repo-content_v2/workshop-Materials/M9-nvidia_nim/.venv


Using CPython 3.12.13
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


notebook env: /home/ubuntu/workshop-materials-gsi/repo-content_v2/workshop-Materials/M9-nvidia_nim/.venv (python /home/ubuntu/workshop-materials-gsi/repo-content_v2/workshop-Materials/M9-nvidia_nim/.venv/bin/python)
installing: torch ...
installing: transformers ...
installing: peft ...
ok: requests
Prerequisites ready.


In [2]:
import torch

# Workshop runs on 1x A100 / H100 / H200 — same recipe; verify GPU here.
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    print(f"GPU: {_p.name} ({_p.total_memory / 2**30:.1f} GiB)")
else:
    print("WARNING: No CUDA GPU detected. Training notebooks will not run; Curator small-corpus paths may still work on CPU.")


GPU: NVIDIA A100 80GB PCIe (79.3 GiB)


In [3]:
import os, subprocess, time, getpass
from pathlib import Path
import requests

# Prompt for NGC API key at runtime (not stored in the notebook).
if not os.environ.get("NGC_API_KEY"):
    _key = getpass.getpass("Enter your NGC API key: ").strip()
    if _key:
        os.environ["NGC_API_KEY"] = _key
if not os.environ.get("NGC_API_KEY"):
    raise ValueError("NGC_API_KEY is required to pull and run NIM containers from nvcr.io")

LOCAL_NIM_CACHE = Path(os.environ["LOCAL_NIM_CACHE"])  # /data/cache/nim (set in cell 1)

NIM_IMAGE = "nvcr.io/nim/nvidia/nvidia-nemotron-nano-9b-v2:latest"
NIM_CONTAINER = "workshop-nim-nano"
NIM_PORT = 8080
NIM_MODEL_ID = "nvidia/nvidia-nemotron-nano-9b-v2"

print("Docker login → nvcr.io …")
subprocess.run(
    ["docker", "login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"],
    input=os.environ["NGC_API_KEY"].encode(),
    check=True,
)

def _nim_ready():
    try:
        return requests.get(f"http://localhost:{NIM_PORT}/v1/models", timeout=5).status_code == 200
    except Exception:
        return False

if _nim_ready():
    print(f"NIM already running on :{NIM_PORT}")
else:
    subprocess.run(["docker", "rm", "-f", NIM_CONTAINER], capture_output=True)
    print("Pulling NIM image (first time: several minutes) …")
    subprocess.check_call(["docker", "pull", NIM_IMAGE])
    print("Starting container …")
    subprocess.check_call([
        "docker", "run", "-d", "--name", NIM_CONTAINER,
        "--gpus", "device=0", "--shm-size", "16g",
        "-e", "NGC_API_KEY", "-e", "NIM_SERVER_PORT=8000",
        "-v", f"{LOCAL_NIM_CACHE}:/opt/nim/.cache",
        "-p", f"{NIM_PORT}:8000", NIM_IMAGE,
    ], env=os.environ.copy())
    for _i in range(40):
        if _nim_ready():
            _r = requests.get(f"http://localhost:{NIM_PORT}/v1/models", timeout=10)
            print("Models:", [m["id"] for m in _r.json().get("data", [])])
            break
        print(f"  waiting for NIM … ({_i + 1}/40)")
        time.sleep(15)
    else:
        raise RuntimeError("NIM not ready — check: docker logs workshop-nim-nano")

os.environ["LOCAL_NIM_URL"] = f"http://localhost:{NIM_PORT}/v1"
print("OpenAI endpoint:", os.environ["LOCAL_NIM_URL"])

from pathlib import Path
from docker_storage import workshop_work_dir, glob_work_paths

NB_DIR = Path.cwd().resolve()
WORK_DIR = workshop_work_dir("M9-nvidia_nim")
M7_NB = NB_DIR.parent / "M7-model_training"
M7_WORK = workshop_work_dir("M7-model_training")
_sft = glob_work_paths(M7_WORK, M7_NB, "sft_checkpoints/**/model/consolidated")
M7_SFT = _sft[-1] if _sft else (M7_NB / "work" / "sft_checkpoints" / "sft-lora-final")
MERGE_DIR = WORK_DIR / "merged_sft"
print('M7 SFT LoRA:', (M7_SFT/'adapter_config.json').exists())


Enter your NGC API key:  ········


Docker login → nvcr.io …


WARNING! Your password will be stored unencrypted in /home/ubuntu/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores



Login Succeeded
Pulling NIM image (first time: several minutes) …
latest: Pulling from nim/nvidia/nvidia-nemotron-nano-9b-v2
65d6848aa6be: Pulling fs layer
ddc9da18b513: Pulling fs layer
4a39b63a208f: Pulling fs layer
8378c496babf: Pulling fs layer
ed0e2082d1bb: Pulling fs layer
b61659d9f609: Pulling fs layer
efaeba21701f: Pulling fs layer
d0ef6a820a7a: Pulling fs layer
b53078d42f1b: Pulling fs layer
9188cf7c8d41: Pulling fs layer
0154c8c7b419: Pulling fs layer
09693755eb54: Pulling fs layer
53afbb9356e9: Pulling fs layer
adedb551814d: Pulling fs layer
d502bdcaf3c6: Pulling fs layer
a0bda0fbe791: Pulling fs layer
36958f672d5a: Pulling fs layer
d5f8005d7dbc: Pulling fs layer
d95e964e9e83: Pulling fs layer
9671628a37fb: Pulling fs layer
a54a593b2866: Pulling fs layer
312f91995407: Pulling fs layer
b400957fb4c3: Pulling fs layer
161b59c42a08: Pulling fs layer
26738a387089: Pulling fs layer
b6504d77d244: Pulling fs layer
4f4fb700ef54: Pulling fs layer
a7ff4ac4e988: Pulling fs layer
b72efa7

## 2. Smoke-test catalog NIM (Nemotron Nano 9B)


In [4]:
import requests
r = requests.post(f'http://localhost:{NIM_PORT}/v1/chat/completions', json={
    'model': NIM_MODEL_ID,
    'messages': [{'role':'user','content':'In one sentence, what is AML structuring?'}],
    'max_tokens': 64,
}, timeout=120)
print(r.json()['choices'][0]['message']['content'])


Okay, the user is asking for a one-sentence definition of AML structuring. Let me start by recalling what AML stands for. AML is Anti-Money Laundering. Structuring in this context probably refers to the practice of breaking down large amounts of money into smaller, less suspicious amounts to avoid detection


## 3. Clean up

Stop and remove the NIM container(s) this notebook started, so the GPU memory and
ports are freed for the next module.

In [5]:
# Stop & remove the NIM container(s) started by this notebook.
# Listed by literal name so this works even if the custom-model cell was skipped.
import subprocess

for _name in ["workshop-nim-nano"]:
    _out = subprocess.run(["docker", "rm", "-f", _name], capture_output=True, text=True)
    if _out.returncode == 0 and _out.stdout.strip():
        print(f"removed: {_name}")
    else:
        print(f"not running / already gone: {_name}")

print("\nRemaining workshop NIM containers:")
subprocess.run(["docker", "ps", "--filter", "name=workshop-",
                "--format", "table {{.Names}}\t{{.Status}}"])

removed: workshop-nim-nano

Remaining workshop NIM containers:
NAMES     STATUS


CompletedProcess(args=['docker', 'ps', '--filter', 'name=workshop-', '--format', 'table {{.Names}}\t{{.Status}}'], returncode=0)

In [ ]:
# --- Download fine-tuned checkpoint from Hugging Face (skip M7.3) ---
import os
import subprocess
from pathlib import Path

from notebook_env import ensure

ensure("huggingface_hub", ["huggingface-hub>=0.26.0"], quiet=True)

HF_TOKEN = "REDACTED_HF_TOKEN"
DOWNLOAD_DIR = Path("/data/sarath/downloaded_checkpoint")

subprocess.run(["hf", "auth", "login", "--token", HF_TOKEN], check=True)

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "hf", "download", "s4sarath/latest_sft_checkpoint",
        "--repo-type", "model",
        "--local-dir", str(DOWNLOAD_DIR),
    ],
    check=True,
)

# Next cell expects a consolidated checkpoint dir (M7 layout or flat HF export).
_consolidated = sorted(DOWNLOAD_DIR.glob("**/model/consolidated"))
_ckpt_dir = _consolidated[-1] if _consolidated else DOWNLOAD_DIR
os.environ["CUSTOM_MODEL_DIR"] = str(_ckpt_dir)

print("Downloaded checkpoint:", _ckpt_dir)
print("Files:", [p.name for p in sorted(_ckpt_dir.iterdir())[:12]], "…" if len(list(_ckpt_dir.iterdir())) > 12 else "")

## 4. Deploy a custom model (Model-Free NIM) from your fine-tuned checkpoint

The catalog NIM above serves a *pre-packaged* model. To serve **your own** trained
checkpoint (e.g. the M7 SFT consolidated model), use a **Model-Free NIM**, which
loads weights from a directory you mount into the container.

Mirrors `gsi-training/9.custom_model_deployment/README.md`, scaled to **1 GPU
(`--tensor-parallel-size 1`)**. Run the HF download cell above if you skipped M7.3.

**Host requirement:** `model-free-nim:2.0.5` needs NVIDIA driver **≥ 580** (CUDA 13).
If your driver is older (e.g. 565), upgrade on the host before running this cell:
`sudo apt install -y nvidia-driver-580 && sudo reboot`

In [ ]:
# --- Custom model: deploy your fine-tuned checkpoint behind a Model-Free NIM ---
# Nemotron-H SFT checkpoint via model-free-nim (vLLM), 1 GPU (TP=1).
# Requires NVIDIA driver >= 580 (CUDA 13) for image tag 2.0.5.

# Workshop SFT checkpoint on /data (fallback: CUSTOM_MODEL_DIR from HF download cell).
_sft_ckpt = glob_work_paths(M7_WORK, M7_NB, "sft_checkpoints/**/model/consolidated")
CUSTOM_MODEL_DIR = str(_sft_ckpt[-1]) if _sft_ckpt else os.environ.get("CUSTOM_MODEL_DIR", "")

CUSTOM_NIM_IMAGE     = "nvcr.io/nim/nvidia/model-free-nim:2.0.5"
CUSTOM_NIM_CONTAINER = "workshop-custom-nim"
CUSTOM_NIM_PORT      = 8088
CUSTOM_SERVED_NAME   = "aml-custom-task-nim-1"
CUSTOM_NIM_MOUNT     = "/model"  # container path for checkpoint (read-only)

assert CUSTOM_MODEL_DIR and Path(CUSTOM_MODEL_DIR).is_dir(), (
    f"No SFT checkpoint found on /data and CUSTOM_MODEL_DIR not set (got {CUSTOM_MODEL_DIR!r}). "
    "Run the HF download cell above, M7.3 sft.ipynb, or export CUSTOM_MODEL_DIR."
)

# model-free-nim:2.0.5 bundles PyTorch+cu130 — needs host driver >= 580.
_drv = subprocess.run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip().split("\n")[0]
_drv_major = int(_drv.split(".")[0]) if _drv else 0
if _drv_major < 580:
    raise RuntimeError(
        f"NVIDIA driver {_drv} is too old for model-free-nim:2.0.5 (needs >= 580). "
        "On the host: sudo apt install -y nvidia-driver-580 && sudo reboot"
    )

print("Checkpoint:", CUSTOM_MODEL_DIR)
print("Pulling Model-Free NIM image (first time: several minutes) …")
subprocess.check_call(["docker", "pull", CUSTOM_NIM_IMAGE])

subprocess.run(["docker", "rm", "-f", CUSTOM_NIM_CONTAINER], capture_output=True)

print("Starting custom NIM (1 GPU, TP=1) …")
subprocess.check_call([
    "docker", "run", "-d",
    "--name", CUSTOM_NIM_CONTAINER,
    "--gpus", '"device=0"',
    "--ipc=host",
    "--shm-size=16g",
    "-p", f"{CUSTOM_NIM_PORT}:8000",              # container listens on 8000
    "-v", f"{CUSTOM_MODEL_DIR}:{CUSTOM_NIM_MOUNT}:ro",
    "-v", f"{LOCAL_NIM_CACHE}:/opt/nim/.cache",
    "-e", f"NIM_MODEL_PATH={CUSTOM_NIM_MOUNT}",
    "-e", f"NIM_SERVED_MODEL_NAME={CUSTOM_SERVED_NAME}",
    "-e", "NIM_TRUST_CUSTOM_CODE=1",
    CUSTOM_NIM_IMAGE,
    "--tensor-parallel-size", "1",
    "--trust-remote-code",                         # required for Nemotron-H custom code
], env=os.environ.copy())

def _custom_ready():
    try:
        return requests.get(f"http://localhost:{CUSTOM_NIM_PORT}/v1/models", timeout=5).status_code == 200
    except Exception:
        return False

for _i in range(60):
    if _custom_ready():
        _r = requests.get(f"http://localhost:{CUSTOM_NIM_PORT}/v1/models", timeout=10)
        print("Custom NIM models:", [m["id"] for m in _r.json().get("data", [])])
        break
    print(f"  waiting for custom NIM … ({_i + 1}/60)")
    time.sleep(15)
else:
    raise RuntimeError(f"Custom NIM not ready — check: docker logs {CUSTOM_NIM_CONTAINER}")

# Smoke-test the custom endpoint.
r = requests.post(f"http://localhost:{CUSTOM_NIM_PORT}/v1/chat/completions", json={
    "model": CUSTOM_SERVED_NAME,
    "messages": [{"role": "user", "content": "In one sentence, what is AML structuring?"}],
    "max_tokens": 64,
}, timeout=120)
print("\nCustom model says:", r.json()["choices"][0]["message"]["content"])

In [9]:
CUSTOM_MODEL_DIR

'/data/sarath/downloaded_checkpoint'

In [10]:
DOWNLOAD_DIR

PosixPath('/data/sarath/downloaded_checkpoint')